# homework 1

q1: pdfs downloaded with `download_books.py`, converted with `convert_books.py`

think python: 9,703 lines

## q2 - chunking

In [1]:
from pathlib import Path

documents = []

for path in sorted(Path('books_text').glob('*.md')):
    text = path.read_text(encoding='utf-8')
    lines = [line for line in text.splitlines() if line.strip()]
    documents.append({
        'source': path.name,
        'content': lines,
    })

len(documents)

7

In [2]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=100, step=50)
len(chunks)

988

In [3]:
python_chunks = [c for c in chunks if c['source'] == 'thinkpython2.md']
len(python_chunks)

189

## q3 - indexing

In [4]:
def prepare_documents(chunks):
    documents = []
    for chunk in chunks:
        doc = chunk.copy()
        doc['content'] = '\n'.join(chunk['content'])
        documents.append(doc)
    return documents

In [5]:
from minsearch import Index

index_docs = prepare_documents(chunks)

index = Index(text_fields=["content"])
index.fit(index_docs)

len(index_docs)

988

## q4 - search

In [6]:
results = index.search("python function definition", num_results=5)

for r in results:
    print(r['source'], r['start'])

PhysicalModelingInMatlab4.md 1800
thinkdsp.md 0
thinkpython2.md 550
PhysicalModelingInMatlab4.md 1750
PhysicalModelingInMatlab4.md 1950


In [7]:
print(results[0]['content'][:500])

| ------ | ------------ | --- | --- |
| total  | = total +    | a;  |     |
end
ans = total
Ifyouwereusinganyofthosevariablenamesbeforecallingthisscript, youmightbesurprised
to find, after running the script, that their values had changed. If you have two scripts that
48 Functions
use the same variable names, you might find that they work separately and then break when
you try to combine them. This kind of interaction is called a collision.
name
As the number of scripts you write increases, and 


## q5 - full rag

using groq (gpt-oss-20b). with 5 results the request is ~8.3k tokens, groq free tier allows max 8k per request -> 413 error. so `num_results=3` to actually run it (4 works for plain, but not with the structured output schema added in q6).

for the answer i count the 5-result prompt with the gpt-4o-mini tokenizer (tiktoken, runs locally)

In [8]:
import os
from openai import OpenAI

openai_client = OpenAI(
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1",
    max_retries=10,
)

In [9]:
import json

instructions = """
You're a course assistant, your task is to answer the QUESTION from the
course students using the provided CONTEXT
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    prompt = prompt_template.format(
        question=question,
        context=context
    ).strip()
    return prompt

def search(question, num_results=3):
    return index.search(question, num_results=num_results)

def llm(user_prompt, instructions, model='openai/gpt-oss-20b'):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text, response.usage

def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer, usage = llm(prompt, instructions)
    return answer, usage

In [10]:
import tiktoken

enc = tiktoken.get_encoding("o200k_base")  # gpt-4o-mini tokenizer

query = "python function definition"
prompt_5 = build_prompt(query, search(query, num_results=5))

len(enc.encode(instructions)) + len(enc.encode(prompt_5))

7915

In [11]:
answer, usage = rag(query)
print(answer)

### How to define a function in Python

```python
def my_func(arg1, arg2=0, *args, **kwargs):
    """Brief description of what the function does.

    Parameters
    ----------
    arg1 : int
        First argument.
    arg2 : int, optional
        Second argument (default 0).
    *args : tuple
        Additional positional arguments.
    **kwargs : dict
        Additional keyword arguments.

    Returns
    -------
    result : int
        Something useful.
    """
    # Body of the function – do the work here
    result = arg1 + arg2
    for a in args:
        result += a
    for key, val in kwargs.items():
        result += val
    return result
```

#### Key points

| Part | What it does | Example |
|------|--------------|---------|
| `def` | Marks the start of a function definition (analogous to MATLAB’s `function` keyword). | `def my_func(x):` |
| `my_func` | Function name (must be a valid identifier). | `my_func` |
| `arg1, arg2=0, *args, **kwargs` | Parameters – mandatory, opti

In [12]:
print('input tokens:', usage.input_tokens)
print('output tokens:', usage.output_tokens)

input tokens: 4653
output tokens: 745


## q6 - structured output

In [13]:
from pydantic import BaseModel, Field
from typing import Literal

class RAGResponse(BaseModel):
    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions")

In [14]:
def llm_structured(user_prompt, instructions, output_type, model='openai/gpt-oss-20b'):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.parse(
        model=model,
        input=messages,
        text_format=output_type
    )

    return response.output_parsed, response.usage

def rag_structured(query, output_type=RAGResponse):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer, usage = llm_structured(prompt, instructions, output_type)
    return answer, usage

In [15]:
s_answer, s_usage = rag_structured(query)

print(s_answer.answer[:300])
print(s_answer.found_answer, s_answer.confidence, s_answer.answer_type)
print(s_answer.followup_questions)

## Defining a function in Python
In Python a **function** is a reusable block of code that takes optional inputs (called *parameters*), performs some computation, and optionally returns a result.

```python
# A simple function that adds two numbers

def add(a, b):
    """Return the sum of a and b.""
True 0.97 explanation
['Do you need help with function scope and variable visibility?', 'Would you like to learn about recursive functions in Python?', 'Are you interested in how to use decorators to modify function behavior?']


In [16]:
print('unstructured input tokens:', usage.input_tokens)
print('structured input tokens:  ', s_usage.input_tokens)
print('difference:', s_usage.input_tokens - usage.input_tokens)

unstructured input tokens: 4653
structured input tokens:   4919
difference: 266


the difference is only the json schema, so it doesn't depend on how many results are in the context (tested with 1 and 3 results, both +266)

answer: 286 (closest to 266)

note: i used groq (openai/gpt-oss-20b) instead of openai gpt-4o-mini. on the groq free tier one request can be max 8k tokens, and the 5-result prompt is ~8.3k, so i had to go down to 3 results to run it. the extra tokens from the schema don't change with the context size, but groq formats the schema a bit differently from openai, so i got 266 instead of the exact option